In [ ]:
!pip install kafka-python pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.1/326.1 kB 8.3 MB/s eta 0:00:00


In [ ]:
# Установка и запуск Kafka в Google Colab

# Шаг 1: Установка зависимостей
!apt-get update -qq
!apt-get install -y -qq default-jre default-jdk wget

# Шаг 2: Скачивание и распаковка Kafka
!wget -q https://archive.apache.org/dist/kafka/3.4.0/kafka_2.13-3.4.0.tgz
!tar -xzf kafka_2.13-3.4.0.tgz
!mv kafka_2.13-3.4.0 kafka

# Шаг 3: Установка Python библиотек
!pip install -q kafka-python pydantic

print("✅ Все зависимости установлены")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libatspi2.0-0:amd64.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../00-libatspi2.0-0_2.44.0-3_amd64.deb ...
Unpacking libatspi2.0-0:amd64 (2.44.0-3) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../01-libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../02-session-migration_0.3.6_amd64.deb ...
Unpacking session-migration (0.3.6) ...
Selecting previously unselected package gsettings-desktop-schemas.
Preparing to unpack .../03-gsettings-desktop-schemas_42.0-1ubuntu1_all.deb ...
Unpacking gsettings-desktop-schemas (42.0-1ubuntu1) ...
Selecting previously unselected

In [ ]:
# Шаг 4: Запуск Zookeeper и Kafka в фоновом режиме
import os
import time
import subprocess
import signal

# Запуск Zookeeper
os.chdir('/content/kafka')
zk_process = subprocess.Popen(
    ['bin/zookeeper-server-start.sh', 'config/zookeeper.properties'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

# Запуск Kafka
kafka_process = subprocess.Popen(
    ['bin/kafka-server-start.sh', 'config/server.properties'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("⏳ Ожидание запуска Kafka (30 секунд)...")
time.sleep(30)
print("✅ Kafka запущена в Colab!")

⏳ Ожидание запуска Kafka (30 секунд)...


In [ ]:
# Шаг 5: Проверка работы Kafka
from kafka import KafkaProducer, KafkaConsumer
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import NoBrokersAvailable
import json
import random
import time
from datetime import datetime
from uuid import uuid4
from typing import Literal, Optional
from pydantic import BaseModel, Field

def check_kafka():
    try:
        admin = KafkaAdminClient(
            bootstrap_servers='localhost:9092',
            request_timeout_ms=5000
        )
        admin.list_topics()
        admin.close()
        print("✅ Kafka работает и доступна!")
        return True
    except Exception as e:
        print(f"❌ Kafka не доступна: {e}")
        return False

check_kafka()

✅ Kafka работает и доступна!


True

In [ ]:
# Шаг 6: Создание топиков
TOPICS = [
    "transactions.raw",
    "transactions.enriched",
    "fraud.alerts",
    "notifications",
    "ledger.events",
    "crm.updates"
]

try:
    admin_client = KafkaAdminClient(
        bootstrap_servers='localhost:9092',
        client_id='topic_creator'
    )

    existing_topics = admin_client.list_topics()

    for topic_name in TOPICS:
        if topic_name not in existing_topics:
            topic = NewTopic(name=topic_name, num_partitions=1, replication_factor=1)
            admin_client.create_topics([topic])
            print(f"✅ Создан топик: {topic_name}")
        else:
            print(f"ℹ️ Топик уже существует: {topic_name}")

    admin_client.close()
    print("\n✅ Все топики готовы!")

except Exception as e:
    print(f"❌ Ошибка: {e}")

✅ Создан топик: transactions.raw
✅ Создан топик: transactions.enriched
✅ Создан топик: fraud.alerts
✅ Создан топик: notifications
✅ Создан топик: ledger.events
✅ Создан топик: crm.updates

✅ Все топики готовы!


In [ ]:
# Шаг 7: Генератор транзакций (Producer)
class Transaction(BaseModel):
    transaction_id: str = Field(default_factory=lambda: str(uuid4()))
    user_id: str
    amount: float
    currency: Literal["RUB", "USD", "EUR"] = "RUB"
    type: Literal["purchase", "transfer", "payment"]
    timestamp: datetime = Field(default_factory=datetime.now)
    merchant_id: Optional[str] = None
    location: str

# Создание продюсера
producer = KafkaProducer(
    bootstrap_servers='localhost:9092',
    value_serializer=lambda v: json.dumps(v, default=str).encode('utf-8')
)

# Данные для генерации
users = [f"user_{i}" for i in range(1, 21)]
merchants = [f"merch_{i}" for i in range(1, 11)]
locations = ["Moscow", "Saint Petersburg", "Novosibirsk", "Kazan", "Sochi"]
types = ["purchase", "transfer", "payment"]

print("Запуск генератора транзакций...\n")

# Отправка 20 транзакций
for i in range(20):
    tx = Transaction(
        user_id=random.choice(users),
        amount=round(random.uniform(10, 500000), 2),
        currency=random.choices(["RUB", "USD", "EUR"], weights=[0.8, 0.15, 0.05])[0],
        type=random.choice(types),
        merchant_id=random.choice(merchants) if random.random() > 0.3 else None,
        location=random.choice(locations)
    )

    tx_dict = tx.model_dump()
    future = producer.send('transactions.raw', value=tx_dict)
    record_metadata = future.get(timeout=10)

    print(f"✅ [{i+1}] {tx.transaction_id[:8]}... | {tx.type} | {tx.amount} {tx.currency} | {tx.user_id} | {tx.location}")
    time.sleep(random.uniform(0.2, 1.0))

producer.close()
print(f"\n✅ Отправлено 20 транзакций в топик 'transactions.raw'")

🚀 Запуск генератора транзакций...

✅ [1] 3431bc5d... | purchase | 155191.71 USD | user_8 | Moscow
✅ [2] 3fdaae39... | payment | 157198.0 RUB | user_12 | Novosibirsk
✅ [3] 0c5f9477... | transfer | 200577.37 RUB | user_10 | Moscow
✅ [4] fb4bb0a9... | purchase | 377271.32 RUB | user_18 | Kazan
✅ [5] 544942d8... | purchase | 212131.7 RUB | user_12 | Sochi
✅ [6] 616a39c5... | purchase | 50585.26 RUB | user_15 | Moscow
✅ [7] bd634291... | transfer | 394639.17 RUB | user_20 | Sochi
✅ [8] 1fbd1321... | transfer | 163234.98 USD | user_15 | Novosibirsk
✅ [9] aa8003ef... | payment | 324242.69 RUB | user_2 | Kazan
✅ [10] b730d5ee... | purchase | 108716.18 EUR | user_8 | Moscow
✅ [11] 494b0437... | purchase | 137708.92 RUB | user_11 | Moscow
✅ [12] af8a3fa3... | payment | 399152.3 RUB | user_2 | Novosibirsk
✅ [13] ef51f9b0... | payment | 239310.6 RUB | user_7 | Saint Petersburg
✅ [14] be4ac091... | transfer | 444684.3 RUB | user_19 | Saint Petersburg
✅ [15] e433d2d5... | transfer | 137450.6 RUB | u

In [ ]:
# Шаг 8: Проверка - читаем отправленные сообщения
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    'transactions.raw',
    bootstrap_servers='localhost:9092',
    auto_offset_reset='earliest',
    enable_auto_commit=True,
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Последние 5 сообщений из топика:\n")
for i, msg in enumerate(consumer):
    if i >= 5:
        break
    print(f"Сообщение {i+1}:")
    print(json.dumps(msg.value, indent=2, ensure_ascii=False))
    print("-" * 50)

consumer.close()

📖 Последние 5 сообщений из топика:

Сообщение 1:
{
  "transaction_id": "3431bc5d-2c57-405b-848a-e0ca43b807e2",
  "user_id": "user_8",
  "amount": 155191.71,
  "currency": "USD",
  "type": "purchase",
  "timestamp": "2026-04-12 19:09:49.436359",
  "merchant_id": "merch_10",
  "location": "Moscow"
}
--------------------------------------------------
Сообщение 2:
{
  "transaction_id": "3fdaae39-2bfb-41b9-abb9-e7e7faa94c8b",
  "user_id": "user_12",
  "amount": 157198.0,
  "currency": "RUB",
  "type": "payment",
  "timestamp": "2026-04-12 19:09:50.156187",
  "merchant_id": null,
  "location": "Novosibirsk"
}
--------------------------------------------------
Сообщение 3:
{
  "transaction_id": "0c5f9477-31df-4336-8bb7-245f6c168a56",
  "user_id": "user_10",
  "amount": 200577.37,
  "currency": "RUB",
  "type": "transfer",
  "timestamp": "2026-04-12 19:09:50.859290",
  "merchant_id": "merch_7",
  "location": "Moscow"
}
--------------------------------------------------
Сообщение 4:
{
  "transa